# SparkClient: Kubeflow Pipelines (KFP) Integration

This notebook demonstrates how to orchestrate a distributed Spark batch job inside a Kubeflow Pipeline using `SparkClient` from the Kubeflow Spark SDK.

### What you will learn:
1. Defining lightweight KFP components that wrap `SparkClient`.
2. Submitting a batch Spark application within a pipeline step.
3. Waiting for job completion and passing the output to downstream steps.
4. Collecting driver pod logs and cleaning up Spark resources.
5. Compiling the pipeline to standard KFP v2 IR YAML.

## 1. Imports and Client Setup

Import the compiler and DSL decorators from `kubeflow.pipelines`.

In [ ]:
from kubeflow.pipelines import compiler, dsl

print("Kubeflow Pipelines imported successfully.")

## 2. Define Pipeline Components

Define components using `@dsl.component`. The first component submits the Spark job and waits for completion; the second retrieves driver logs and deletes the job resources.

In [ ]:
@dsl.component(
    base_image="python:3.11",
    packages_to_install=["kubeflow[spark]"],
)
def submit_spark_job(
    file_source: str,
    namespace: str = "default",
    num_executors: int = 1,
    executor_cpu: str = "1",
    executor_memory: str = "512Mi",
    timeout: int = 600,
) -> str:
    """Submit a Spark batch job and wait for completion."""
    from kubeflow.common.types import KubernetesBackendConfig
    from kubeflow.spark import FileJob, SparkClient, SparkJobStatus

    backend_config = KubernetesBackendConfig(namespace=namespace)
    client = SparkClient(backend_config=backend_config)

    job_name = client.submit_job(
        job=FileJob(file_source=file_source),
        num_executors=num_executors,
        resources_per_executor={
            "cpu": executor_cpu,
            "memory": executor_memory,
        },
    )
    print(f"Submitted Spark batch job: {job_name}")

    completed_job = client.wait_for_job_status(
        name=job_name,
        status={SparkJobStatus.COMPLETED},
        timeout=timeout,
    )
    print(f"Spark job {job_name} finished with status: {completed_job.status}")
    return job_name

In [ ]:
@dsl.component(
    base_image="python:3.11",
    packages_to_install=["kubeflow[spark]"],
)
def inspect_and_cleanup_spark_job(
    job_name: str,
    namespace: str = "default",
) -> str:
    """Retrieve driver pod logs and delete the Spark batch job."""
    from kubeflow.common.types import KubernetesBackendConfig
    from kubeflow.spark import SparkClient

    backend_config = KubernetesBackendConfig(namespace=namespace)
    client = SparkClient(backend_config=backend_config)

    logs = list(client.get_job_logs(name=job_name))
    log_summary = f"Retrieved {len(logs)} log lines from driver pod for job {job_name}."
    print(log_summary)

    client.delete_job(name=job_name)
    print(f"Cleaned up Spark batch job: {job_name}")
    return log_summary

## 3. Define the Pipeline Graph

Compose the components into a sequential workflow using `@dsl.pipeline`.

In [ ]:
@dsl.pipeline(
    name="spark-batch-pipeline",
    description="Kubeflow Pipeline orchestrating a distributed Spark batch job via SparkClient.",
)
def spark_pipeline(
    file_source: str = "https://raw.githubusercontent.com/kubeflow/sdk/main/examples/spark/spark_job.py",
    namespace: str = "default",
    num_executors: int = 1,
):
    spark_task = submit_spark_job(
        file_source=file_source,
        namespace=namespace,
        num_executors=num_executors,
    )

    cleanup_task = inspect_and_cleanup_spark_job(
        job_name=spark_task.output,
        namespace=namespace,
    )
    cleanup_task.after(spark_task)

## 4. Compile the Pipeline to KFP v2 YAML

Use `compiler.Compiler()` to generate the standard pipeline IR package.

In [ ]:
compiled_yaml = "spark_batch_pipeline.yaml"
compiler.Compiler().compile(
    pipeline_func=spark_pipeline,
    package_path=compiled_yaml,
)
print(f"Successfully compiled pipeline to: {compiled_yaml}")

## 5. (Optional) Submit to a Live Kubeflow Cluster

When running in an environment with access to a Kubeflow deployment, submit the pipeline using `PipelinesClient`.

In [ ]:
# from kubeflow.pipelines import PipelinesClient
# client = PipelinesClient()
# run = client.run(pipeline=spark_pipeline, name="spark-pipeline-run")
# print(f"Submitted pipeline run ID: {run.id}")